In [22]:
import os
import json
import subprocess
import requests
import urllib3

# Suppress SSL verification warnings (equivalent to curl -k)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- CONFIGURATION & ENV SETUP ---
# Grabs MAAS_URL from environment variable, falls back to your logged URL if empty
MAAS_URL = os.environ.get("MAAS_URL", "maas.apps.rosa.poc-andyr.70qk.p3.openshiftapps.com")

try:
    # Dynamically grab the OpenShift token exactly like $(oc whoami -t)
    OC_TOKEN = subprocess.check_output(["oc", "whoami", "-t"], text=True).strip()
except (subprocess.CalledProcessError, FileNotFoundError):
    # Fallback placeholder if running where 'oc' CLI isn't installed/logged in
    OC_TOKEN = os.environ.get("OC_TOKEN", "YOUR_OPENSHIFT_TOKEN")

print(f"Configured MAAS_URL: {MAAS_URL}")


# -------------------------------------------------------------------------
# STEP 1: Generate MaaS API Key
# -------------------------------------------------------------------------
print("\n[Step 1/3] Generating API Key...")
keygen_url = f"https://{MAAS_URL}/maas-api/v1/api-keys"

headers_step1 = {
    "Authorization": f"Bearer {OC_TOKEN}",
    "Content-Type": "application/json"
}
payload_step1 = {
    "name": "test-key", 
    "subscription": "simulator-premium", 
    "expiresIn": "1h"
}

response_step1 = requests.post(
    keygen_url, 
    headers=headers_step1, 
    json=payload_step1, 
    verify=False
)
response_step1.raise_for_status()

# Equivalent to storing the key parsed via jq
api_key = response_step1.json()["key"]
print("Key generated successfully.")


Configured MAAS_URL: maas.apps.rosa.poc-andyr.70qk.p3.openshiftapps.com

[Step 1/3] Generating API Key...
Key generated successfully.


In [23]:
# -------------------------------------------------------------------------
# STEP 2: List Models & Extract Target URL
# -------------------------------------------------------------------------
print("\n[Step 2/3] Querying Available Models...")
models_url = f"https://{MAAS_URL}/v1/models"

headers_step2 = {
    "Authorization": f"Bearer {api_key}"
}

response_step2 = requests.get(
    models_url, 
    headers=headers_step2, 
    verify=False
)
response_step2.raise_for_status()
models_json = response_step2.json()

print(response_step2.json())

# Equivalent to jq -r '.data[0].url' and getting the model ID dynamically
model_url = models_json["data"][0]["url"]
model_id = models_json["data"][0]["id"]

print(f"-> Target Model ID: {model_id}")
print(f"-> Extracted Model URL: {model_url}")


[Step 2/3] Querying Available Models...
{'data': [{'id': 'gpt-oss-20b', 'created': 1782137891, 'object': 'model', 'owned_by': 'llm/gpt-oss-20b', 'kind': 'LLMInferenceService', 'url': 'https://maas.apps.rosa.poc-andyr.70qk.p3.openshiftapps.com/llm/gpt-oss-20b', 'ready': True, 'modelDetails': {'description': 'OpenAI gpt-oss-20b on vLLM with NVIDIA GPU', 'displayName': 'OpenAI gpt-oss-20b'}, 'subscriptions': [{'name': 'gpt-oss-20b-premium', 'displayName': 'gpt-oss-20b Premium Tier', 'description': 'Premium tier: 10000 tokens/min for premium users'}]}], 'object': 'list'}
-> Target Model ID: gpt-oss-20b
-> Extracted Model URL: https://maas.apps.rosa.poc-andyr.70qk.p3.openshiftapps.com/llm/gpt-oss-20b


In [24]:
# -------------------------------------------------------------------------
# STEP 3: Submit Chat Completion Request
# -------------------------------------------------------------------------
print("\n[Step 3/3] Sending Chat Completion...")
chat_url = f"{model_url}/v1/chat/completions"

headers_step3 = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}
payload_step3 = {
    "model": model_id,
    "messages": [{"role": "user", "content": "What is a fish?"}]
}

response_step3 = requests.post(
    chat_url, 
    headers=headers_step3, 
    json=payload_step3, 
    verify=False
)
response_step3.raise_for_status()

print("\n--- Final API Response ---")
print(json.dumps(response_step3.json(), indent=2))


[Step 3/3] Sending Chat Completion...

--- Final API Response ---
{
  "id": "chatcmpl-74added6-56f1-4ee2-90b7-00716ae0783a",
  "object": "chat.completion",
  "created": 1782137894,
  "model": "gpt-oss-20b",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "A **fish** is any aquatic vertebrate that:\n\n| Feature | Explanation |\n|---------|-------------|\n| **Habitat** | Lives in water (freshwater or marine). |\n| **Respiration** | Breathe oxygen through gills (or a similar respiratory organ). |\n| **Body plan** | Typically has a streamlined, fusiform shape, fins for locomotion, scales (except for some basal groups), and a skeletal system made of bone or cartilage. |\n| **Physiology** | Cold\u2011blooded (ectothermic), usually oviparous (egg\u2011laying), though a few species give birth to live young. |\n| **Taxonomy** | Falls under the clade *Osteichthyes* (bony fish) and *Chondrichthyes* (cartilaginous fish). Together, along wi